# MIRI imaging photometry: LTT 1445 A (real data)

This tutorial produces F1500W light curves of LTT 1445 A from JWST program 9234
(observation 1) with aperture and 2-D Gaussian-process PSF photometry. It is **not executed**
when the documentation is built: it downloads about 1 GB and needs CRDS reference files.

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import transitspectroscopy as ts

In [ ]:
ts.jwst.download(pid=9234, obs_num='1', data_product='rateints')
files = sorted(glob.glob('JWSTdata/jw09234001001_03101_00001-seg*_mirimage_rateints.fits'))
print(files)

## Reduce one visit

`extract_photometry` runs the official JWST `calwebb_image2` calibration on each segment, cuts out
the target, estimates an annular background, and measures aperture and optimal photometry.

In [ ]:
result = ts.miri.extract_photometry(files, outputfolder='miri_ltt1445a', input_product='rateints',
                             aperture_radius=5., training_integrations=12)
print(sorted(result))

## Light curves

In [ ]:
times = result['times']
flux = result['flux']            # columns: aperture sum, GP PSF photometry
plt.figure(figsize=(10, 4))
for k, method in enumerate(('aperture sum', 'GP PSF')):
    plt.plot((times - times[0]) * 24, flux[:, k] / np.nanmedian(flux[:, k]) + .005 * k, '.', ms=1,
             label=method)
plt.xlabel('Hours since start'); plt.ylabel('Relative flux (offset)'); plt.legend();

See {doc}`../guides/miri` and {doc}`../validation/miri-report`. Injection tests show that
GP photometry recovers shallow (300 ppm) eclipses about 30% low; compare both methods before
interpreting shallow events.